# Out-of-distribution test — does the probe still match the SAE on fresh, off-corpus text?

The in-distribution result (median per-concept AUC 0.90) is on held-out documents from the *training corpus mix* (web/code/CoT/math/chat). This notebook tests the honest next question **systematically**: take text from domains the probe never trained on — **medical abstracts and legal text** — run the real Gemma-2-9B + SAE to get ground-truth firings, run the probe, and recompute the *same* metrics. If the median holds near 0.90 on genuinely off-corpus text, OOD generalization is demonstrated, not assumed.

Needs a GPU (A100) with `gemma-2-9b` access. Upload **`golf_final.zip`** (the probe).

In [1]:
# 1 · install
!pip -q install transformer-lens sae-lens scikit-learn sentencepiece datasets huggingface_hub
!pip -q uninstall -y torchcodec torchvision torchaudio

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.1/145.1 kB 13.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 57.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.7/311.7 kB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.6/42.6 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 285.3/285.3 kB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 236.9/236.9 kB 29.2 MB/s eta 0:00:00


In [2]:
# 2 · config + HF login (gated gemma)
MODEL_NAME, SAE_RELEASE, SAE_ID = "gemma-2-9b", "gemma-scope-9b-pt-res-canonical", "layer_20/width_16k/canonical"
PROBE="/content/golf_final"; ACT_FLOOR=1.0; SPAN_MIN,SPAN_MAX,MAXPERDOC=12,80,10
OOD_DOCS=400
import os,shutil
if not os.path.isdir(PROBE) and os.path.exists("/content/golf_final.zip"): shutil.unpack_archive("/content/golf_final.zip",PROBE)
from huggingface_hub import login; login()

In [3]:
# 3 · load teacher (Gemma + SAE) and the probe
import os; os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True")
import json,re,gc,numpy as np,torch,torch.nn as nn,math
from transformer_lens import HookedTransformer
from sae_lens import SAE
DEV="cuda"
model=HookedTransformer.from_pretrained(MODEL_NAME,device=DEV,dtype=torch.bfloat16)
_r=SAE.from_pretrained(SAE_RELEASE,SAE_ID,device=DEV); sae=(_r[0] if isinstance(_r,tuple) else _r).to(torch.bfloat16)
_m=getattr(sae.cfg,"metadata",None); HOOK=getattr(sae.cfg,"hook_name",None) or getattr(_m,"hook_name",None) or "blocks.20.hook_resid_post"
PAD=model.tokenizer.pad_token_id or model.tokenizer.eos_token_id
import sentencepiece as spm
sp=spm.SentencePieceProcessor(model_file=f"{PROBE}/mega_tok.model")
ck=torch.load(f"{PROBE}/mega_golf_probe.pth",map_location="cpu",weights_only=False); cfg=ck["cfg"]
V=np.array(ck["V"]); N_V=len(V); MU=np.asarray(ck["mu"],np.float32); SD=np.asarray(ck["sd"],np.float32)
UNI,BIGH,CDIM,FDIM,NH,NL,MAXLEN=cfg["UNI"],cfg["BIG_H"],cfg["CDIM"],cfg["FDIM"],cfg["NHEAD"],cfg["NLAYER"],cfg["MAXLEN"]
lut=np.full(16384,-1); lut[V]=np.arange(N_V)                       # map full-SAE latent id -> kept index
class Stock(nn.Module):
    def __init__(s):
        super().__init__(); d,e=CDIM,FDIM
        s.uni,s.big=nn.Embedding(UNI,e),nn.Embedding(BIGH,e); s.proj=nn.Linear(e,d,bias=False); s.pos=nn.Embedding(MAXLEN,d)
        s.enc=nn.TransformerEncoder(nn.TransformerEncoderLayer(d,NH,4*d,batch_first=True,dropout=0.1),NL); s.head=nn.Linear(d,N_V)
    def span(s,u,b,m):
        x=s.proj(s.uni(u)+s.big(b))+s.pos(torch.arange(u.shape[1])[None]); t=s.head(s.enc(x,src_key_padding_mask=~m))
        mk=t.masked_fill(~m.unsqueeze(-1),-1e30); return torch.logsumexp(mk,1)-torch.log(m.sum(1,keepdim=True).float())
probe=Stock().eval(); probe.load_state_dict(ck["state_dict"])
print("teacher + probe loaded ·",N_V,"concepts")

config.json:   0%|          | 0.00/856 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/39.1k [00:00<?, ?B/s]

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/464 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

Loaded pretrained model gemma-2-9b into HookedTransformer


layer_20/width_16k/average_l0_68/params.(…):   0%|          | 0.00/470M [00:00<?, ?B/s]

teacher + probe loaded · 2048 concepts


In [4]:
# 4 · stream OOD corpora (NOT in training: medical + legal) -> spans
from datasets import load_dataset
WS,SENT=re.compile(r"[ \t]+"),re.compile(r"(?<=[.!?])\s+")
def norm(t): return WS.sub(" ",str(t).replace("\r","")).strip()
OOD={"medical":[("ccdv/pubmed-summarization",None,"article"),("pubmed_qa","pqa_labeled","context")],
     "legal":[("pile-of-law/pile-of-law","all","text"),("lex_glue","ledgar","text")]}
def pick(rec,keys):
    for k in ([keys] if isinstance(keys,str) else keys):
        v=rec.get(k)
        if isinstance(v,str) and len(v)>300: return v
        if isinstance(v,dict):
            for vv in v.values():
                if isinstance(vv,str) and len(vv)>300: return vv
    return ""
docs=[]
for dom,srcs in OOD.items():
    got=0
    for name,cfg_,key in srcs:
        try: ds=load_dataset(name,cfg_,split="train",streaming=True)
        except Exception as e: print("  skip",name,type(e).__name__); continue
        for rec in ds:
            t=norm(pick(rec,key))
            if len(t)>400: docs.append((dom,t[:8000])); got+=1
            if got>=OOD_DOCS//2: break
        if got: break
    print(f"  {dom}: {got} docs")
spans=[]
for di,(dom,text) in enumerate(docs):
    buf=[]; n=0
    for sent in SENT.split(text):
        buf.append(sent.strip()); cand=" ".join(buf)
        ids=model.tokenizer(cand)["input_ids"]
        if ids and ids[0]==model.tokenizer.bos_token_id: ids=ids[1:]
        if len(ids)>=SPAN_MIN:
            spans.append({"dom":dom,"text":cand,"ids":ids[:SPAN_MAX]}); buf=[]; n+=1
            if n>=MAXPERDOC: break
print("OOD spans:",len(spans))

README.md:   0%|          | 0.00/3.80k [00:00<?, ?B/s]

  medical: 200 docs


README.md:   0%|          | 0.00/25.6k [00:00<?, ?B/s]

pile-of-law.py:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

  skip pile-of-law/pile-of-law RuntimeError


README.md:   0%|          | 0.00/34.1k [00:00<?, ?B/s]

  skip lex_glue HfUriError
  legal: 0 docs
OOD spans: 1997


In [5]:
# 5 · harvest the REAL SAE firings on OOD spans (ground truth), restricted to the probe's kept concepts
gc.collect(); torch.cuda.empty_cache()
n=len(spans); Y=np.zeros((n,N_V),bool)
order=sorted(range(n),key=lambda i:len(spans[i]["ids"])); BS=16
with torch.no_grad():
    for b0 in range(0,n,BS):
        idxs=order[b0:b0+BS]; L=max(len(spans[i]["ids"]) for i in idxs)
        ids=torch.full((len(idxs),L),PAD,dtype=torch.long)
        for r,i in enumerate(idxs): ids[r,:len(spans[i]["ids"])]=torch.tensor(spans[i]["ids"])
        _,cache=model.run_with_cache(ids.to(DEV),names_filter=HOOK,return_type=None)
        f=sae.encode(cache[HOOK]).float(); del cache
        for r,i in enumerate(idxs):
            m=f[r,:len(spans[i]["ids"])]
            fired=(m>ACT_FLOOR).any(0).nonzero().squeeze(-1).cpu().numpy()
            kept=lut[fired]; Y[i,kept[kept>=0]]=True
        if b0%(BS*20)==0: print(f"  harvested {b0}/{n}")
print("SAE firings harvested · mean concepts/span",Y.sum(1).mean().round(1))

  harvested 0/1997
  harvested 320/1997
  harvested 640/1997
  harvested 960/1997
  harvested 1280/1997
  harvested 1600/1997
  harvested 1920/1997
SAE firings harvested · mean concepts/span 170.8


In [6]:
# 6 · run the probe on the OOD spans (its own tokenizer)
def bg(ids): return np.concatenate([[0],(36313*ids[1:]^27191*ids[:-1])%BIGH]).astype(np.int64)
@torch.no_grad()
def score():
    S=np.zeros((n,N_V),np.float32); od=sorted(range(n),key=lambda i:len(sp.encode(spans[i]["text"])[:MAXLEN]) or 1)
    for b0 in range(0,n,128):
        ch=od[b0:b0+128]; idl=[sp.encode(spans[i]["text"])[:MAXLEN] or [0] for i in ch]; L=max(len(x) for x in idl)
        u=torch.zeros(len(ch),L,dtype=torch.long); b=torch.zeros(len(ch),L,dtype=torch.long); m=torch.zeros(len(ch),L,dtype=torch.bool)
        for r,ids in enumerate(idl):
            k=len(ids); u[r,:k]=torch.tensor(ids); b[r,:k]=torch.tensor(bg(np.asarray(ids))); m[r,:k]=True
        s=probe.span(u,b,m).numpy()
        for r,i in enumerate(ch): S[i]=s[r]
    return S
Z=(score()-MU)/SD; print("probe scored OOD spans")

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


probe scored OOD spans


In [7]:
# 7 · SAME metrics as in-distribution — does recovery hold off-corpus?
from sklearn.metrics import roc_auc_score
auc=[roc_auc_score(Y[:,c],Z[:,c]) for c in range(N_V) if 0<Y[:,c].sum()<n]
auc=np.array(auc)
top5=np.mean([(Y[i][np.argsort(-Z[i])[:5]]).mean() for i in range(n) if Y[i].sum()>0])
print("="*70)
print(f"OUT-OF-DISTRIBUTION (medical + legal, {n} spans, real SAE ground truth):")
print(f"  per-concept AUC: median {np.median(auc):.3f}  ·  {(auc>0.6).mean():.0%} above noise  ·  {(auc<=0.55).mean():.0%} at chance")
print(f"  top-5 precision: {top5:.3f}  ({top5*20:.1f} of 20 shown are real SAE firings)")
print(f"  concepts measured: {len(auc)}")
print("-"*70)
print(f"  IN-DISTRIBUTION baseline was: median AUC 0.90 · top-5 precision 0.956")
print(f"  => OOD drop: AUC {0.903-np.median(auc):+.3f} · top-5 {0.956-top5:+.3f}")
print("="*70)
# per-domain
for dom in ["medical","legal"]:
    idx=[i for i in range(n) if spans[i]["dom"]==dom]
    da=[roc_auc_score(Y[idx,c],Z[idx,c]) for c in range(N_V) if 0<Y[idx][:,c].sum()<len(idx)]
    print(f"  {dom}: median AUC {np.median(da):.3f} ({len(idx)} spans)")

OUT-OF-DISTRIBUTION (medical + legal, 1997 spans, real SAE ground truth):
  per-concept AUC: median 0.745  ·  92% above noise  ·  2% at chance
  top-5 precision: 0.938  (18.8 of 20 shown are real SAE firings)
  concepts measured: 1978
----------------------------------------------------------------------
  IN-DISTRIBUTION baseline was: median AUC 0.90 · top-5 precision 0.956
  => OOD drop: AUC +0.158 · top-5 +0.018
  medical: median AUC 0.745 (1997 spans)
  legal: median AUC nan (0 spans)


/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:3596: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [8]:
# 8 · LEGAL OOD — harvest a legal corpus separately, score, and combine with medical
from datasets import load_dataset
from sklearn.metrics import roc_auc_score
LEGAL=[("coastalcph/lex_glue","ledgar","text"),
       ("pile-of-law/pile-of-law","courtlistener_opinions","text"),
       ("lex_glue","ledgar","text")]
def norm2(t): return re.sub(r"[ \t]+"," ",str(t).replace("\r","")).strip()
ldocs=[]
for name,cfg_,key in LEGAL:
    try: ds=load_dataset(name,cfg_,split="train",streaming=True)
    except Exception as e: print(" skip",name,type(e).__name__); continue
    for rec in ds:
        t=norm2(rec.get(key,""))
        if len(t)>300: ldocs.append(t[:8000])
        if len(ldocs)>=200: break
    if ldocs: print("legal source:",name,"·",len(ldocs),"docs"); break
# spans
lspans=[]
SENT2=re.compile(r"(?<=[.!?])\s+")
for text in ldocs:
    buf=[]; k=0
    for s in SENT2.split(text):
        buf.append(s.strip()); cand=" ".join(buf)
        ids=model.tokenizer(cand)["input_ids"]
        if ids and ids[0]==model.tokenizer.bos_token_id: ids=ids[1:]
        if len(ids)>=SPAN_MIN:
            lspans.append({"text":cand,"ids":ids[:SPAN_MAX]}); buf=[]; k+=1
            if k>=MAXPERDOC: break
nl=len(lspans); print("legal spans:",nl)
# harvest SAE firings (ground truth)
gc.collect(); torch.cuda.empty_cache()
Yl=np.zeros((nl,N_V),bool); od=sorted(range(nl),key=lambda i:len(lspans[i]["ids"])); BS=8
with torch.no_grad():
    for b0 in range(0,nl,BS):
        ix=od[b0:b0+BS]; L=max(len(lspans[i]["ids"]) for i in ix)
        ids=torch.full((len(ix),L),PAD,dtype=torch.long)
        for r,i in enumerate(ix): ids[r,:len(lspans[i]["ids"])]=torch.tensor(lspans[i]["ids"])
        _,cache=model.run_with_cache(ids.to(DEV),names_filter=HOOK,return_type=None)
        f=sae.encode(cache[HOOK]).float(); del cache
        for r,i in enumerate(ix):
            fired=(f[r,:len(lspans[i]["ids"])]>ACT_FLOOR).any(0).nonzero().squeeze(-1).cpu().numpy()
            kept=lut[fired]; Yl[i,kept[kept>=0]]=True
        del f
# score probe
Sl=np.zeros((nl,N_V),np.float32); od2=sorted(range(nl),key=lambda i:len(sp.encode(lspans[i]["text"])[:MAXLEN]) or 1)
with torch.no_grad():
    for b0 in range(0,nl,128):
        ch=od2[b0:b0+128]; idl=[sp.encode(lspans[i]["text"])[:MAXLEN] or [0] for i in ch]; L=max(len(x) for x in idl)
        u=torch.zeros(len(ch),L,dtype=torch.long); b=torch.zeros(len(ch),L,dtype=torch.long); m=torch.zeros(len(ch),L,dtype=torch.bool)
        for r,ids in enumerate(idl):
            k=len(ids); u[r,:k]=torch.tensor(ids); b[r,:k]=torch.tensor(bg(np.asarray(ids))); m[r,:k]=True
        s=probe.span(u,b,m).numpy()
        for r,i in enumerate(ch): Sl[i]=s[r]
Zl=(Sl-MU)/SD
# metrics: legal alone + combined with medical (Y, Z from cell 5-6)
la=np.array([roc_auc_score(Yl[:,c],Zl[:,c]) for c in range(N_V) if 0<Yl[:,c].sum()<nl])
lt=np.mean([Yl[i][np.argsort(-Zl[i])[:5]].mean() for i in range(nl) if Yl[i].sum()>0])
Yc=np.vstack([Y,Yl]); Zc=np.vstack([Z,Zl]); nc=Yc.shape[0]
ca=np.array([roc_auc_score(Yc[:,c],Zc[:,c]) for c in range(N_V) if 0<Yc[:,c].sum()<nc])
ct=np.mean([Yc[i][np.argsort(-Zc[i])[:5]].mean() for i in range(nc) if Yc[i].sum()>0])
print("="*60)
print(f"LEGAL   : median AUC {np.median(la):.3f} · top-5 {lt:.3f} · {nl} spans")
print(f"MEDICAL : median AUC {np.median([roc_auc_score(Y[:,c],Z[:,c]) for c in range(N_V) if 0<Y[:,c].sum()<len(Y)]):.3f}")
print(f"COMBINED OOD (medical+legal, {nc} spans): median AUC {np.median(ca):.3f} · top-5 {ct:.3f} · {(ca>0.6).mean():.0%} above noise")
print(f"in-distribution baseline: AUC 0.90 · top-5 0.956")
print("="*60)


README.md:   0%|          | 0.00/34.1k [00:00<?, ?B/s]

legal source: coastalcph/lex_glue · 200 docs
legal spans: 505
LEGAL   : median AUC 0.727 · top-5 0.939 · 505 spans
MEDICAL : median AUC 0.745
COMBINED OOD (medical+legal, 2502 spans): median AUC 0.750 · top-5 0.938 · 92% above noise
in-distribution baseline: AUC 0.90 · top-5 0.956


In [9]:
# 9 · OOD baselines — probe vs bge (naive) vs random, SAE = ground truth (the full stack)
!pip -q install sentence-transformers
from sentence_transformers import SentenceTransformer
from sklearn.metrics import roc_auc_score
bemb=SentenceTransformer("BAAI/bge-small-en-v1.5",device="cpu")   # tiny; CPU keeps VRAM for gemma
caps=ck["captions"]
texts=[s["text"] for s in spans]+[s["text"] for s in lspans]      # combined OOD spans (medical+legal)
Et=bemb.encode(texts,normalize_embeddings=True,batch_size=128)
Ec=bemb.encode(caps,normalize_embeddings=True,batch_size=256)
bge=Et@Ec.T                                                       # zero-shot cosine [spans, concepts]
Yc=np.vstack([Y,Yl]); Zc=np.vstack([Z,Zl]); nc=Yc.shape[0]
pa=[]; ba=[]
for c in range(N_V):
    y=Yc[:,c]
    if 0<y.sum()<nc: pa.append(roc_auc_score(y,Zc[:,c])); ba.append(roc_auc_score(y,bge[:,c]))
pa,ba=np.array(pa),np.array(ba)
print("="*60)
print(f"OUT-OF-DISTRIBUTION (medical+legal, {nc} spans) — the full stack:")
print(f"  random chance      : 0.500")
print(f"  bge (naive embed)  : {np.median(ba):.3f}")
print(f"  OUR PROBE          : {np.median(pa):.3f}")
print(f"  SAE                : 1.000  (ground truth / the target)")
print(f"  probe beats bge on {(pa>ba).mean():.0%} of concepts, even OOD")
print("="*60)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

OUT-OF-DISTRIBUTION (medical+legal, 2502 spans) — the full stack:
  random chance      : 0.500
  bge (naive embed)  : 0.551
  OUR PROBE          : 0.750
  SAE                : 1.000  (ground truth / the target)
  probe beats bge on 95% of concepts, even OOD


In [10]:
# 10 · OOD nuance — the SHAPE of the drop: which concepts hold, which fail, and what kind
caps=ck["captions"]
oa=np.full(N_V,np.nan)
for c in range(N_V):
    y=Yc[:,c]
    if 0<y.sum()<nc: oa[c]=roc_auc_score(y,Zc[:,c])
ok=~np.isnan(oa); a=oa[ok]; idx=np.where(ok)[0]
print(f"OOD per-concept AUC — the distribution ({ok.sum()} concepts), not the median:")
for lo,hi,lab in [(0.9,1.01,"held strong 0.9+"),(0.75,0.9,"held 0.75-0.9"),(0.6,0.75,"weakened 0.6-0.75"),(0,0.6,"failed <0.6")]:
    m=(a>=lo)&(a<hi); print(f"  {lab:20s}: {m.sum():4d} ({m.mean():.0%})")
print("\nHELD UP best OOD (these transfer):")
for c in idx[np.argsort(-oa[idx])[:10]]: print(f"  {oa[c]:.2f}  {caps[c][:62]}")
print("\nFAILED worst OOD (these were tied to the training domains):")
for c in idx[np.argsort(oa[idx])[:10]]: print(f"  {oa[c]:.2f}  {caps[c][:62]}")


OOD per-concept AUC — the distribution (2004 concepts), not the median:
  held strong 0.9+    :  405 (20%)
  held 0.75-0.9       :  596 (30%)
  weakened 0.6-0.75   :  847 (42%)
  failed <0.6         :  156 (8%)

HELD UP best OOD (these transfer):
  1.00  The latent detects phrases that indicate a continuation or a p
  1.00  The latent detects the pronoun 'you' used in direct address or
  1.00  This latent detects mathematical expressions involving fractio
  1.00  This latent detects questions, especially those seeking clarif
  1.00  Detects the asterisk character used as a wildcard or placehold
  1.00  The latent detects hypothetical or conditional statements, oft
  1.00  Detects phrases referring to the recipient's possessions or re
  1.00  Mathematical expressions involving variables like x1, x2, and 
  1.00  Python code defining classes, methods, and URL patterns, often
  1.00  Detects Python class definitions and method definitions, espec

FAILED worst OOD (these were tied to the t